In [ ]:
# 1) Load a subset of the 20 Newsgroups dataset
from sklearn.datasets import fetch_20newsgroups

categories = [
    'alt.atheism',
    'comp.graphics',
    'sci.med',
    'soc.religion.christian'
]

twenty_train = fetch_20newsgroups(
    subset='train',
    categories=categories,
    shuffle=True,
    random_state=42
)
print("Loaded %d documents" % len(twenty_train.data))
print("Target names:", twenty_train.target_names)

# 2) Peek at the first document and its target
print("\n".join(twenty_train.data[0].split("\n")[:3]))
print("→ Category:", twenty_train.target_names[twenty_train.target[0]])

# 3) Show the first 15 targets as names
for t in twenty_train.target[:15]:
    print(twenty_train.target_names[t])


# 4) Turn raw text into count-vectors
from sklearn.feature_extraction.text import CountVectorizer

count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(twenty_train.data)
print("Count matrix shape:", X_train_counts.shape)
print("Index of 'algorithm' in vocab:", count_vect.vocabulary_.get('algorithm'))


# 5) Convert counts to TF (term-frequencies) or TF–IDF
from sklearn.feature_extraction.text import TfidfTransformer

# first just TF
tf_transformer = TfidfTransformer(use_idf=False).fit(X_train_counts)
X_train_tf = tf_transformer.transform(X_train_counts)

# then full TF–IDF
tfidf_transformer = TfidfTransformer().fit(X_train_counts)
X_train_tfidf = tfidf_transformer.transform(X_train_counts)


# 6) Train a Multinomial Naive Bayes classifier
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB().fit(X_train_tfidf, twenty_train.target)

# make predictions on new docs
docs_new = ['God is love', 'OpenGL on the GPU is fast']
X_new_counts = count_vect.transform(docs_new)
X_new_tfidf   = tfidf_transformer.transform(X_new_counts)

predicted = clf.predict(X_new_tfidf)
for doc, category in zip(docs_new, predicted):
    print(f"'{doc}' => {twenty_train.target_names[category]}")


# 7) Build a full pipeline: vectorizer → transformer → classifier
from sklearn.pipeline import Pipeline

text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

text_clf.fit(twenty_train.data, twenty_train.target)


# 8) Evaluate on the test subset
twenty_test = fetch_20newsgroups(
    subset='test',
    categories=categories,
    shuffle=True,
    random_state=42
)
docs_test = twenty_test.data
predicted = text_clf.predict(docs_test)

from sklearn import metrics
print("Accuracy:", np.mean(predicted == twenty_test.target))
print(metrics.classification_report(
    twenty_test.target, predicted,
    target_names=twenty_test.target_names
))
print("Confusion matrix:\n", metrics.confusion_matrix(
    twenty_test.target, predicted
))


In [ ]:
# 1) Load a subset of the 20 Newsgroups dataset
from sklearn.datasets import fetch_20newsgroups

categories = [
    'alt.atheism',
    'comp.graphics',
    'sci.med',
    'soc.religion.christian'
]

twenty_train = fetch_20newsgroups(
    subset='train',
    categories=categories,
    shuffle=True,
    random_state=42
)
print("Loaded %d documents" % len(twenty_train.data))
print("Target names:", twenty_train.target_names)

# 2) Peek at the first document and its target
print("\n".join(twenty_train.data[0].split("\n")[:3]))
print("→ Category:", twenty_train.target_names[twenty_train.target[0]])

# 3) Show the first 15 targets as names
for t in twenty_train.target[:15]:
    print(twenty_train.target_names[t])

# 4) Turn raw text into count-vectors
from sklearn.feature_extraction.text import CountVectorizer

count_vect = CountVectorizer()
X_train_counts = count_vect.fit_transform(twenty_train.data)
print("Count matrix shape:", X_train_counts.shape)
print("Index of 'algorithm' in vocab:", count_vect.vocabulary_.get('algorithm'))


# 5) Convert counts to TF (term-frequencies) or TF–IDF
from sklearn.feature_extraction.text import TfidfTransformer

# first just TF
tf_transformer = TfidfTransformer(use_idf=False).fit(X_train_counts)
X_train_tf = tf_transformer.transform(X_train_counts)

# then full TF–IDF
tfidf_transformer = TfidfTransformer().fit(X_train_counts)
X_train_tfidf = tfidf_transformer.transform(X_train_counts)

# 6) Train a Multinomial Naive Bayes classifier
from sklearn.naive_bayes import MultinomialNB

clf = MultinomialNB().fit(X_train_tfidf, twenty_train.target)

# make predictions on new docs
docs_new = ['God is love', 'OpenGL on the GPU is fast']
X_new_counts = count_vect.transform(docs_new)
X_new_tfidf   = tfidf_transformer.transform(X_new_counts)

predicted = clf.predict(X_new_tfidf)
for doc, category in zip(docs_new, predicted):
    print(f"'{doc}' => {twenty_train.target_names[category]}")


# 7) Build a full pipeline: vectorizer → transformer → classifier
from sklearn.pipeline import Pipeline

text_clf = Pipeline([
    ('vect', CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf', MultinomialNB()),
])

text_clf.fit(twenty_train.data, twenty_train.target)


# 8) Evaluate on the test subset
twenty_test = fetch_20newsgroups(
    subset='test',
    categories=categories,
    shuffle=True,
    random_state=42
)
docs_test = twenty_test.data
predicted = text_clf.predict(docs_test)

from sklearn import metrics
print("Accuracy:", np.mean(predicted == twenty_test.target))
print(metrics.classification_report(
    twenty_test.target, predicted,
    target_names=twenty_test.target_names
))
print("Confusion matrix:\n", metrics.confusion_matrix(
    twenty_test.target, predicted
))

In [ ]:
# 1) Predict on new documents with the existing CountVectorizer → TfidfTransformer → MultinomialNB chain

docs_new = ['God is love', 'OpenGL on the GPU is fast']
X_new_counts = count_vect.transform(docs_new)
X_new_tfidf   = tfidf_transformer.transform(X_new_counts)

predicted = clf.predict(X_new_tfidf)

for doc, category in zip(docs_new, predicted):
    print("%r => %s" % (doc, 
        twenty_train.target_names[category]))
# Output:
# 'God is love'             => soc.religion.christian
# 'OpenGL on the GPU is fast' => comp.graphics

# 2) Wrap the whole pipeline up in a single Pipeline object
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB

text_clf = Pipeline([
    ('vect',  CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf',   MultinomialNB()),
])

# fit to the training data
text_clf.fit(twenty_train.data, twenty_train.target)

# 3) Evaluate performance on the test split
import numpy as np
from sklearn.datasets import fetch_20newsgroups

twenty_test = fetch_20newsgroups(
    subset='test',
    categories=categories,
    shuffle=True,
    random_state=42
)
docs_test = twenty_test.data
predicted = text_clf.predict(docs_test)
print("NB accuracy:", np.mean(predicted == twenty_test.target))
# e.g. 0.8349

# 4) Try a linear SVM via SGDClassifier
from sklearn.linear_model import SGDClassifier

text_clf = Pipeline([
    ('vect',  CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf',   SGDClassifier(
        loss='hinge',
        penalty='l2',
        alpha=1e-3,
        max_iter=5,
        tol=None,
        random_state=42
    )),
])

text_clf.fit(twenty_train.data, twenty_train.target)
predicted = text_clf.predict(docs_test)
print("SVM accuracy:", np.mean(predicted == twenty_test.target))
# e.g. 0.91


# 5) Try a Decision Tree
from sklearn.tree import DecisionTreeClassifier

text_clf = Pipeline([
    ('vect',  CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf',   DecisionTreeClassifier(random_state=0)),
])

text_clf.fit(twenty_train.data, twenty_train.target)
predicted = text_clf.predict(docs_test)
print("DT accuracy:", np.mean(predicted == twenty_test.target))
# e.g. 0.705


# 6) And finally Logistic Regression
from sklearn.linear_model import LogisticRegression

text_clf = Pipeline([
    ('vect',  CountVectorizer()),
    ('tfidf', TfidfTransformer()),
    ('clf',   LogisticRegression(random_state=42)),
])

text_clf.fit(twenty_train.data, twenty_train.target)
predicted = text_clf.predict(docs_test)
print("Logistic accuracy:", np.mean(predicted == twenty_test.target))
# e.g. 0.8975
